# Role 2 – Baseline Models (Elliptic Bitcoin)

Run the notebook **top to bottom**, in Google Colab or locally. It downloads the public Elliptic dataset from KaggleHub, reads the team split from `tests/elliptic_split.csv` and scores the final model with the Role 4 evaluator (`src/role4_evaluation.py`).

- **Colab:** copy this repository to Google Drive as `MyDrive/ML-project-main`. Results go to `MyDrive/Role2_Results`.
- **Locally:** start Jupyter from the repository root or from `notebooks/`. Results go to `outputs/role2/`, which git ignores.

**Methods:** train on 1–29 and validate on 30–34 to choose the model. Then retrain the chosen Random Forest on labelled periods 1–34, once per seed in `SEEDS`, and evaluate on labelled periods 35–49 with a fixed 0.50 threshold. The graph degree uses same-period edges. That is a batch-graph assumption, not proof that the information is available online.

**Outputs:** one prediction file and one evaluator folder per seed, plus a combined `comparison/summary.csv` (mean ± std over seeds). Each run writes to a new timestamped folder, so re-running never overwrites earlier results. Do not commit private tokens or local datasets.

In [1]:
import sys
from pathlib import Path

# Where the repository is and where results go
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = Path("/content/drive/MyDrive/ML-project-main")
    RESULTS_DIR = Path("/content/drive/MyDrive/Role2_Results")
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    RESULTS_DIR = REPO / "outputs" / "role2"

# Seeds for the final Random Forest (the Role 3 notebook uses the same seeds)
SEEDS = [0, 1, 2, 3, 4]

assert (REPO / "src" / "role4_evaluation.py").exists(), f"Repository not found at {REPO}"
assert (REPO / "tests" / "elliptic_split.csv").exists(), "Split file not found"
print("Repository:", REPO)
print("Results:", RESULTS_DIR)

Repository: <repo>
Results: <repo>\outputs\role2


In [2]:

%pip -q install kagglehub

import kagglehub
from pathlib import Path

dataset_path = kagglehub.dataset_download(
    "ellipticco/elliptic-data-set"
)

print("Dataset location:", dataset_path)

for file in Path(dataset_path).rglob("*"):
    if file.is_file():
        print(file.relative_to(dataset_path))



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: <venv>\Scripts\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


<venv>\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset location: ~/.cache\kagglehub\datasets\ellipticco\elliptic-data-set\versions\1
elliptic_bitcoin_dataset\elliptic_txs_classes.csv
elliptic_bitcoin_dataset\elliptic_txs_edgelist.csv
elliptic_bitcoin_dataset\elliptic_txs_features.csv


In [3]:

import pandas as pd
from pathlib import Path

folder = Path(dataset_path) / "elliptic_bitcoin_dataset"

features = pd.read_csv(
    folder / "elliptic_txs_features.csv",
    header=None
)

classes = pd.read_csv(
    folder / "elliptic_txs_classes.csv"
)

edges = pd.read_csv(
    folder / "elliptic_txs_edgelist.csv"
)

print("Features:", features.shape)
print("Classes:", classes.shape)
print("Edges:", edges.shape)

print("\nFirst 5 transactions:")
display(features.head())

print("\nClass distribution:")
print(classes["class"].value_counts())


Features: (203769, 167)
Classes: (203769, 2)
Edges: (234355, 2)

First 5 transactions:


,0,1,2,3,4,5,6,7,8,9,...,157,158,159,160,161,162,163,164,165,166
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.562153,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112,...,0.947382,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,...,0.670883,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.577099,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,...,-0.511871,-0.400422,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117



Class distribution:
class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64


In [4]:
import pandas as pd

split = pd.read_csv(
    REPO / "tests" / "elliptic_split.csv",
    dtype={"txId": str, "class": str}
)

assert len(split) == 203769
assert split["txId"].is_unique

print("SPLIT LOADED SUCCESSFULLY")
print("Total rows:", len(split))
print(split["split"].value_counts())

SPLIT LOADED SUCCESSFULLY
Total rows: 203769
split
train    136265
test      67504
Name: count, dtype: int64


In [5]:

import numpy as np
import pandas as pd

# Prepare original Elliptic features
feature_data = features.copy()

feature_data.columns = (
    ["txId", "time_step"] +
    [f"feature_{i}" for i in range(1, 166)]
)

feature_data["txId"] = feature_data["txId"].astype(str)

# Join with the official team split
data = feature_data.merge(
    split,
    on=["txId", "time_step"],
    how="inner",
    validate="one_to_one"
)

assert len(data) == len(split), "Some transactions are missing"

# Use the 165 anonymized numerical features
feature_columns = [
    f"feature_{i}" for i in range(1, 166)
]

# Keep only transactions with known labels
labelled = data[data["class"].isin(["licit", "illicit"])].copy()
labelled["target"] = (labelled["class"] == "illicit").astype(int)

# Development: train 1-29, validation 30-34
train = labelled[
    (labelled["split"] == "train") &
    (labelled["time_step"] <= 29)
].copy()

validation = labelled[
    (labelled["split"] == "train") &
    (labelled["time_step"].between(30, 34))
].copy()

test = labelled[
    labelled["split"] == "test"
].copy()

print("Training:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))
print("Features:", len(feature_columns))

print("\nTraining classes:")
print(train["class"].value_counts())


Training: 26381
Validation: 3513
Test: 16670
Features: 165

Training classes:
class
licit      23510
illicit     2871
Name: count, dtype: int64


In [6]:

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

# Create the training and validation inputs
X_train = train[feature_columns]
y_train = train["target"]

X_val = validation[feature_columns]
y_val = validation["target"]

# Build the baseline model
logistic_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=0
    )
)

# Train only on the training period
logistic_model.fit(X_train, y_train)

# Predict illicit probabilities on validation data
val_scores = logistic_model.predict_proba(X_val)[:, 1]
val_predictions = (val_scores >= 0.50).astype(int)

print("LOGISTIC REGRESSION - VALIDATION")
print("Precision:", round(precision_score(y_val, val_predictions, zero_division=0), 4))
print("Recall:", round(recall_score(y_val, val_predictions, zero_division=0), 4))
print("F1:", round(f1_score(y_val, val_predictions, zero_division=0), 4))
print("Average Precision:", round(average_precision_score(y_val, val_scores), 4))



LOGISTIC REGRESSION - VALIDATION
Precision: 0.3559
Recall: 0.978
F1: 0.5219
Average Precision: 0.5844


In [7]:

%pip -q install xgboost

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score
)

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        class_weight="balanced",
        random_state=0,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.1,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=0,
        n_jobs=2
    )
}

validation_results = []

# Include our existing Logistic Regression
validation_results.append({
    "Model": "Logistic Regression",
    "Precision": precision_score(y_val, val_predictions),
    "Recall": recall_score(y_val, val_predictions),
    "F1": f1_score(y_val, val_predictions),
    "AP": average_precision_score(y_val, val_scores)
})

for name, model in models.items():
    model.fit(X_train, y_train)
    scores = model.predict_proba(X_val)[:, 1]
    predictions = (scores >= 0.50).astype(int)

    validation_results.append({
        "Model": name,
        "Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_val, predictions, zero_division=0
        ),
        "AP": average_precision_score(y_val, scores)
    })

results_df = pd.DataFrame(validation_results)
display(results_df.round(4))



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: <venv>\Scripts\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


,Model,Precision,Recall,F1,AP
0,Logistic Regression,0.3559,0.9780,0.5219,0.5844
1,Random Forest,0.9676,0.9611,0.9643,0.9867
2,XGBoost,0.9772,0.7986,0.8790,0.9835


In [8]:

# Graph feature: within-period degree
node_periods = feature_data[["txId", "time_step"]].copy()

edge_pairs = edges.iloc[:, :2].copy()
edge_pairs.columns = ["source", "target"]
edge_pairs = edge_pairs.astype(str)

edge_pairs = edge_pairs.merge(
    node_periods.rename(columns={
        "txId": "source",
        "time_step": "source_period"
    }),
    on="source",
    how="left",
    validate="many_to_one"
)

edge_pairs = edge_pairs.merge(
    node_periods.rename(columns={
        "txId": "target",
        "time_step": "target_period"
    }),
    on="target",
    how="left",
    validate="many_to_one"
)

assert edge_pairs[
    ["source_period", "target_period"]
].notna().all().all()

# Keep edges within the same time period
same_period = edge_pairs[
    edge_pairs["source_period"] ==
    edge_pairs["target_period"]
]

all_endpoints = pd.concat([
    same_period["source"],
    same_period["target"]
])

degree = all_endpoints.value_counts()

data["graph_degree"] = (
    data["txId"].map(degree).fillna(0).astype(int)
)

print("Graph feature created successfully!")
print("Edges in the same period:", len(same_period))
print("Maximum degree:", data["graph_degree"].max())
print("Average degree:", round(data["graph_degree"].mean(), 4))


Graph feature created successfully!
Edges in the same period: 234355
Maximum degree: 473
Average degree: 2.3002


In [9]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score,
    f1_score, average_precision_score
)

# Add the graph feature to our 165 original features
graph_features = feature_columns + ["graph_degree"]

# Prepare training and validation data with graph information
graph_train = data.loc[train.index]
graph_val = data.loc[validation.index]

X_train_graph = graph_train[graph_features]
X_val_graph = graph_val[graph_features]

# Train the same Random Forest configuration
graph_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    class_weight="balanced",
    random_state=0,
    n_jobs=-1
)

graph_model.fit(X_train_graph, y_train)

graph_scores = graph_model.predict_proba(X_val_graph)[:, 1]
graph_predictions = (graph_scores >= 0.50).astype(int)

graph_result = pd.DataFrame([{
    "Model": "Random Forest + Degree",
    "Precision": precision_score(y_val, graph_predictions, zero_division=0),
    "Recall": recall_score(y_val, graph_predictions, zero_division=0),
    "F1": f1_score(y_val, graph_predictions, zero_division=0),
    "AP": average_precision_score(y_val, graph_scores)
}])

comparison = pd.concat([
    results_df[results_df["Model"] == "Random Forest"],
    graph_result
], ignore_index=True)

display(comparison.round(4))


,Model,Precision,Recall,F1,AP
0,Random Forest,0.9676,0.9611,0.9643,0.9867
1,Random Forest + Degree,0.9611,0.9628,0.9620,0.9849


In [10]:

# Verify the chronological split
assert train["time_step"].between(1, 29).all()
assert validation["time_step"].between(30, 34).all()
assert test["time_step"].between(35, 49).all()

# Verify there are no overlapping transactions
train_ids = set(train["txId"])
val_ids = set(validation["txId"])
test_ids = set(test["txId"])

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

# Verify all model inputs are numerical
assert all(
    pd.api.types.is_numeric_dtype(data[col])
    for col in feature_columns + ["graph_degree"]
)

# Verify that all expected labelled rows are present
assert len(train) + len(validation) == 29894
assert len(test) == 16670

# Check the feature schema and graph periods
assert len(feature_columns) == 165
assert (edge_pairs["source_period"] ==
        edge_pairs["target_period"]).all()

print("ALL STRUCTURAL CHECKS PASSED")
print("Training periods: 1-29")
print("Validation periods: 30-34")
print("Test periods: 35-49")
print("No overlapping transaction IDs")
print("All 165 numerical features verified")
print("Graph edges stay within the same period")


ALL STRUCTURAL CHECKS PASSED
Training periods: 1-29
Validation periods: 30-34
Test periods: 35-49
No overlapping transaction IDs
All 165 numerical features verified
Graph edges stay within the same period


In [11]:
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier

# All labelled training transactions, periods 1-34
final_train = labelled[
    labelled["split"] == "train"
].copy()

assert final_train["time_step"].between(1, 34).all()
assert test["time_step"].between(35, 49).all()

# A new folder per notebook run, so earlier results are never overwritten
run_dir = RESULTS_DIR / datetime.now().strftime("run_%Y%m%d_%H%M%S")
run_dir.mkdir(parents=True)

test_scores = {}
prediction_files = {}

for seed in SEEDS:
    final_model = RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        class_weight="balanced",
        random_state=seed,
        n_jobs=-1
    )
    final_model.fit(final_train[feature_columns], final_train["target"])

    # Export probabilities, not predicted labels
    illicit_index = list(final_model.classes_).index(1)
    test_scores[seed] = final_model.predict_proba(
        test[feature_columns]
    )[:, illicit_index]

    prediction_files[seed] = run_dir / f"random_forest_seed{seed}_predictions.csv"
    pd.DataFrame({
        "txId": test["txId"].astype(str).values,
        "score_illicit": test_scores[seed]
    }).to_csv(prediction_files[seed], index=False)

print("FINAL MODELS TRAINED SUCCESSFULLY")
print("Training examples:", len(final_train))
print("Seeds:", SEEDS)
print("Prediction files saved to", run_dir)

FINAL MODELS TRAINED SUCCESSFULLY
Training examples: 29894
Seeds: [0, 1, 2, 3, 4]
Prediction files saved to <repo>\outputs\role2\run_20261116_173001


In [12]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

# These labels are used only for evaluation
y_test = test["target"].to_numpy()

rows = []
for seed, scores in test_scores.items():
    predictions = (scores >= 0.50).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_test, predictions, labels=[0, 1]
    ).ravel()
    rows.append({
        "Seed": seed,
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
        "AP": average_precision_score(y_test, scores),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn
    })

seed_results = pd.DataFrame(rows)
print("FINAL RANDOM FOREST - TEST RESULTS PER SEED")
display(seed_results.round(4))
print("\nMean over seeds:")
display(seed_results[["Precision", "Recall", "F1", "AP"]].agg(["mean", "std"]).round(4))

FINAL RANDOM FOREST - TEST RESULTS PER SEED


,Seed,Precision,Recall,F1,AP,TP,FP,FN,TN
0,0,0.7692,0.7322,0.7502,0.7861,793,238,290,15349
1,1,0.7733,0.7276,0.7498,0.7846,788,231,295,15356
2,2,0.7960,0.7276,0.7603,0.7879,788,202,295,15385
3,3,0.7678,0.7267,0.7467,0.7883,787,238,296,15349
4,4,0.7693,0.7359,0.7522,0.7910,797,239,286,15348



Mean over seeds:


,Precision,Recall,F1,AP
mean,0.7751,0.7300,0.7518,0.7876
std,0.0118,0.0039,0.0051,0.0024


In [13]:
# Expected known-label test transaction IDs
expected_ids = set(
    split.loc[
        (split["split"] == "test") &
        (split["class"].isin(["licit", "illicit"])),
        "txId"
    ].astype(str)
)
assert len(expected_ids) == 16670

for seed, path in prediction_files.items():
    predictions_df = pd.read_csv(path, dtype={"txId": str})
    assert set(predictions_df["txId"]) == expected_ids
    assert predictions_df["txId"].is_unique
    assert predictions_df["score_illicit"].notna().all()
    assert predictions_df["score_illicit"].between(0, 1).all()

print("PREDICTION FILE VALIDATION PASSED")
print("All 16,670 IDs match the shared test split, for every seed")
print("No duplicate or missing transaction IDs")
print("All scores are between 0 and 1")

PREDICTION FILE VALIDATION PASSED
All 16,670 IDs match the shared test split, for every seed
No duplicate or missing transaction IDs
All scores are between 0 and 1


In [14]:
import subprocess

# Install the team's evaluation dependencies
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "-q", "-r", str(REPO / "requirements-evaluation.txt")
    ],
    check=True
)

# Run the Role 4 evaluator once per seed
evaluation_dirs = []
for seed, path in prediction_files.items():
    outdir = run_dir / "role4" / f"random_forest_seed{seed}"
    subprocess.run(
        [
            sys.executable, "-m", "src.role4_evaluation",
            "--split-file", "tests/elliptic_split.csv",
            "--predictions", str(path),
            "--model", "RandomForest",
            "--seed", str(seed),
            "--threshold", "0.50",
            "--outdir", str(outdir)
        ],
        cwd=str(REPO),
        check=True
    )
    evaluation_dirs.append(outdir)

print("OFFICIAL EVALUATION COMPLETED")

OFFICIAL EVALUATION COMPLETED


In [15]:
# Combine the seeds: mean and std, plus F1 before and after the step-43 shift
subprocess.run(
    [
        sys.executable, "-m", "src.role4_compare",
        *map(str, evaluation_dirs),
        "--outdir", str(run_dir / "comparison")
    ],
    cwd=str(REPO),
    check=True
)

print("Results saved to", run_dir)
display(pd.read_csv(run_dir / "comparison" / "summary.csv").T)

Results saved to <repo>\outputs\role2\run_20261116_173001


,0
model,RandomForest
n_runs,5
precision_illicit_mean,0.775107
precision_illicit_std,0.011837
recall_illicit_mean,0.730009
recall_illicit_std,0.00395
f1_illicit_mean,0.751834
f1_illicit_std,0.005109
average_precision_mean,0.787591
average_precision_std,0.002415
